# Inference Pipeline: Apply ** Model to New Data

This notebook demonstrates how to use the trained ** model for inference:
- Load the trained model
- Create a prediction pipeline
- Apply to unlabeled chunks from the database
- Batch inference on larger datasets


In [1]:
import pickle
import numpy as np
import pandas as pd
import torch
# DuckDB no longer needed - using CSV files
from transformers import BartForSequenceClassification, AutoTokenizer, pipeline
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


PyTorch version: 2.5.1+cu121
CUDA available: True


## 1. Load Model and Configuration


In [ ]:
# Load configuration
with open('data/config.pkl', 'rb') as f:
    config = pickle.load(f)

with open('data/label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

print("Configuration:")
print(f"  Model: {config['model_name']}")
print(f"  Number of labels: {config['num_labels']}")
print(f"  Labels: {list(label_encoder.classes_)}")


In [ ]:
# Load trained model
model = BartForSequenceClassification.from_pretrained('models/best_model')
tokenizer = AutoTokenizer.from_pretrained('models/best_model')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

print(f"\n✅ Model loaded and ready for inference")
print(f"Device: {device}")


## 2. Create Inference Functions


In [ ]:
def predict_frame(text, return_probabilities=False):
    """
    Predict the frame label for a single text chunk.
    
    Args:
        text (str): Input text to classify
        return_probabilities (bool): If True, return all class probabilities
    
    Returns:
        dict: Prediction results including label, confidence, and optionally probabilities
    """
    # Tokenize
    inputs = tokenizer(
        text, 
        return_tensors='pt', 
        padding=True, 
        truncation=True, 
        max_length=config['max_length']
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    
    # Get prediction
    pred_idx = np.argmax(probs)
    pred_label = label_encoder.inverse_transform([pred_idx])[0]
    confidence = probs[pred_idx]
    
    result = {
        'predicted_label': pred_label,
        'confidence': float(confidence),
    }
    
    if return_probabilities:
        result['all_probabilities'] = {
            label_encoder.inverse_transform([i])[0]: float(prob)
            for i, prob in enumerate(probs)
        }
    
    return result

print("✅ Inference function defined")


In [ ]:
def predict_batch(texts, batch_size=16, show_progress=True):
    """
    Predict frame labels for a batch of texts.
    
    Args:
        texts (list): List of text chunks to classify
        batch_size (int): Batch size for processing
        show_progress (bool): Show progress bar
    
    Returns:
        pd.DataFrame: Predictions with labels and confidence scores
    """
    predictions = []
    confidences = []
    
    iterator = range(0, len(texts), batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="Predicting")
    
    with torch.no_grad():
        for i in iterator:
            batch_texts = texts[i:i + batch_size]
            
            # Tokenize batch
            inputs = tokenizer(
                batch_texts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=config['max_length']
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Predict
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            
            # Extract predictions and confidences
            batch_preds = np.argmax(probs, axis=1)
            batch_confs = probs[np.arange(len(batch_preds)), batch_preds]
            
            predictions.extend(batch_preds)
            confidences.extend(batch_confs)
    
    # Convert to labels
    pred_labels = label_encoder.inverse_transform(predictions)
    
    return pd.DataFrame({
        'predicted_label': pred_labels,
        'confidence': confidences
    })

print("✅ Batch prediction function defined")


## 3. Test on Sample Texts


In [ ]:
# Test examples
sample_texts = [
    "The European Union regulations are harming British businesses and causing economic damage to our industries.",
    "We must regain control of our borders and our laws from Brussels bureaucrats who don't represent our interests.",
    "Brexit will have devastating consequences for ordinary families who rely on EU funding and cooperation.",
    "This is fundamentally a question of sovereignty and democratic accountability to the British people.",
]

print("Testing on sample texts:")
print("=" * 80)

for i, text in enumerate(sample_texts, 1):
    result = predict_frame(text, return_probabilities=True)
    print(f"\nSample {i}:")
    print(f"Text: {text}")
    print(f"Predicted Frame: {result['predicted_label']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"All probabilities:")
    for label, prob in sorted(result['all_probabilities'].items(), key=lambda x: x[1], reverse=True):
        print(f"  {label:20s}: {prob:.4f}")
    print("-" * 80)


## 4. Load Unlabeled Chunks from Database


In [ ]:
# Load validation data as demo for inference (simulates unlabeled data)
# In production, replace this with your actual unlabeled data CSV
val_df = pd.read_csv('training_data_lang_15pct_2.csv')

# Use a sample for demo purposes
unlabeled_df = val_df.sample(n=min(100, len(val_df)), random_state=42).reset_index(drop=True)

# Add minimal metadata (chunk_id if not present)
if 'chunk_id' not in unlabeled_df.columns:
    unlabeled_df['chunk_id'] = [f'chunk_{i:06d}' for i in range(len(unlabeled_df))]

print(f"✅ Loaded {len(unlabeled_df)} chunks for inference demo")
print(f"   (Using validation data sample - replace with your unlabeled data)")
print(f"\nSample:")
if 'chunk_id' in unlabeled_df.columns:
    print(unlabeled_df[['chunk_id', 'chunk_text']].head()[[unlabeled_df.columns[0]]])


## 5. Run Batch Predictions


In [ ]:
# Run batch predictions
print(f"Running predictions on {len(unlabeled_df)} chunks...")
predictions_df = predict_batch(unlabeled_df['chunk_text'].tolist(), batch_size=16)

# Combine with original data
unlabeled_df['predicted_frame'] = predictions_df['predicted_label']
unlabeled_df['confidence'] = predictions_df['confidence']

print(f"\n✅ Predictions complete!")
print(f"\nPredicted frame distribution:")
print(unlabeled_df['predicted_frame'].value_counts())
print(f"\nAverage confidence: {unlabeled_df['confidence'].mean():.4f}")


## 6. Examine Predictions


In [ ]:
# Display predictions for each frame type
print("\n" + "=" * 80)
print("SAMPLE PREDICTIONS BY FRAME")
print("=" * 80)

for frame in unlabeled_df['predicted_frame'].unique():
    frame_samples = unlabeled_df[unlabeled_df['predicted_frame'] == frame].head(2)
    
    if len(frame_samples) > 0:
        print(f"\n{'='*80}")
        print(f"FRAME: {frame}")
        print(f"{'='*80}")
        
        for idx, row in frame_samples.iterrows():
            print(f"\nChunk ID: {row['chunk_id']}")
            print(f"Speaker: {row['speaker_name']}")
            print(f"Date: {row['debate_date']}")
            print(f"Confidence: {row['confidence']:.4f}")
            print(f"Text: {row['chunk_text'][:300]}...")
            print("-" * 80)


## 7. High Confidence Predictions


In [ ]:
# Show high confidence predictions
high_confidence = unlabeled_df[unlabeled_df['confidence'] > 0.9].sort_values('confidence', ascending=False)

print(f"\nHigh confidence predictions (> 0.9): {len(high_confidence)}")
print("\nTop 5 most confident predictions:")
print("=" * 80)

for idx, row in high_confidence.head(5).iterrows():
    print(f"\nPredicted Frame: {row['predicted_frame']}")
    print(f"Confidence: {row['confidence']:.4f}")
    print(f"Speaker: {row['speaker_name']}")
    print(f"Text: {row['chunk_text'][:200]}...")
    print("-" * 80)


## 8. Save Predictions


In [ ]:
# Save predictions
unlabeled_df.to_csv('results/unlabeled_predictions.csv', index=False)
print(f"\n✅ Predictions saved to results/unlabeled_predictions.csv")

# Summary statistics
summary = {
    'total_predictions': len(unlabeled_df),
    'frame_distribution': unlabeled_df['predicted_frame'].value_counts().to_dict(),
    'avg_confidence': float(unlabeled_df['confidence'].mean()),
    'high_confidence_count': len(high_confidence),
    'low_confidence_count': len(unlabeled_df[unlabeled_df['confidence'] < 0.5])
}

with open('results/inference_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)

print("✅ Summary saved to results/inference_summary.pkl")


## 9. Usage Example: Predict on Custom Text


In [ ]:
# Example: How to use the model for your own text
print("\n" + "=" * 80)
print("USAGE EXAMPLE: Custom Text Prediction")
print("=" * 80)

custom_text = """
The withdrawal from the European Union will give us the freedom to negotiate our own 
trade deals and make our own decisions without interference from Brussels. This is about 
taking back control of our country and our destiny.
"""

result = predict_frame(custom_text.strip(), return_probabilities=True)

print(f"\nYour text: {custom_text.strip()[:200]}...")
print(f"\nPredicted Frame: {result['predicted_label']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"\nAll class probabilities:")
for label, prob in sorted(result['all_probabilities'].items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(prob * 50)
    print(f"  {label:20s}: {prob:.4f} {bar}")

print("\n" + "=" * 80)
print("To use this model on your own text, simply call:")
print("  result = predict_frame('your text here')")
print("=" * 80)


## 10. Inference Summary


In [ ]:
print("\n" + "=" * 80)
print("INFERENCE SUMMARY")
print("=" * 80)

print(f"\n📊 Predictions:")
print(f"   - Total chunks processed: {len(unlabeled_df)}")
print(f"   - Average confidence: {unlabeled_df['confidence'].mean():.4f}")

print(f"\n🏷️  Frame Distribution:")
for frame, count in unlabeled_df['predicted_frame'].value_counts().items():
    pct = count / len(unlabeled_df) * 100
    print(f"   - {frame:20s}: {count:3d} ({pct:5.1f}%)")

print(f"\n🎯 Confidence Levels:")
print(f"   - High confidence (>0.9): {len(unlabeled_df[unlabeled_df['confidence'] > 0.9])}")
print(f"   - Medium confidence (0.5-0.9): {len(unlabeled_df[(unlabeled_df['confidence'] >= 0.5) & (unlabeled_df['confidence'] <= 0.9)])}")
print(f"   - Low confidence (<0.5): {len(unlabeled_df[unlabeled_df['confidence'] < 0.5])}")

print(f"\n💾 Files Saved:")
print(f"   - results/unlabeled_predictions.csv")
print(f"   - results/inference_summary.pkl")

print("\n✅ Inference pipeline complete!")
print("=" * 80)

print("\n📝 Next Steps:")
print("   1. Review the predictions in results/unlabeled_predictions.csv")
print("   2. Use predict_frame() function for single text predictions")
print("   3. Use predict_batch() for large-scale batch predictions")
print("   4. Fine-tune the model with more data if needed")
